# Held Out Publication Analysis

In [2]:
import pandas as pd
import numpy as np

import json
with open('config.json', 'r') as f:
    config = json.load(f)
    
from helpers.sementic_recoding import drop_by_threshold,recode_semantic_missingness
from helpers.modeling import (
    prepare_data,
    identify_column_types,
    create_preprocessor,
    evaluate_model,
    run_grid_search,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR, NuSVR  # regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(config['filepath'], encoding="latin-1", header=[0, 1])
print(f"Initial shape: {df.shape}")

Initial shape: (2189, 92)


In [3]:
df.columns = df.columns.get_level_values(1)

column_names = config['column_names']
df.columns = column_names

df = df.iloc[1:].reset_index(drop=True)
df = df.drop(columns=['mix_id', 'country', 'reference_details', 'publish_year'])

for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.isna().sum() == df[col].isna().sum():
        df[col] = converted

print(f"Cleaned shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
df['paper_reference'] = df['paper_reference'].ffill()
df.head()

Cleaned shape: (2188, 88)
Total columns: 88


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,shrinkage_standard,shrinkage_28d,shrinkage_56d,freeze_thaw_standard,freeze_thaw_cycles,rcpt_standard,rcpt,resistivity_standard,surface_resistivity,paper_reference
0,839.0,Type I/II low-alkali portland cement,NaN,104.0,104.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,270.0,371.0,ASTM C666,105.0,ASTM C1202,165.0,AASHTO T 358,386.0,Ref-1-data
1,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,242.0,317.0,ASTM C666,106.0,ASTM C1202,142.0,AASHTO T 358,424.0,Ref-1-data
2,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,223.0,290.0,ASTM C666,106.0,ASTM C1202,136.0,AASHTO T 358,463.0,Ref-1-data
3,839.0,Type I/II low-alkali portland cement,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ref-1-data
4,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,231.0,286.0,ASTM C666,106.0,ASTM C1202,135.0,AASHTO T 358,437.0,Ref-1-data


In [4]:
# Remove rows with missing target variable
df = df.dropna(subset=['cs_28d'])
print(f"Rows with 28-day compressive strength data: {len(df)}")

# Define columns to drop (focusing on cs_28d as target)
drop_cols = config["drop_cols"]

# Drop unnecessary columns
df = df.drop(columns=drop_cols)
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")



Rows with 28-day compressive strength data: 2073
Final shape after filtering: (2073, 44)
Remaining columns: 44


In [5]:
df_cleaned = pd.read_csv(config["initial_cleaned_filepath"], index_col=0)
df_cleaned['paper_reference'] = df['paper_reference'].values
df_cleaned.iloc[1]
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")
df_cleaned.head(20)

Final shape after filtering: (2073, 44)
Remaining columns: 44


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d,paper_reference
0,839.0,OPC_I,NaN,104.0,104.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,56.50,Standard Curing,NaN,NaN,NaN,135.0,Ref-1-data
1,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,132.0,Ref-1-data
2,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,122.5,Ref-1-data
3,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,116.0,Ref-1-data
4,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,134.0,Ref-1-data
5,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,131.5,Ref-1-data
6,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,67.80,Standard Curing,NaN,NaN,NaN,130.5,Ref-1-data
7,839.0,OPC_I,NaN,104.0,83.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,134.5,Ref-1-data
8,839.0,OPC_I,NaN,104.0,62.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,132.5,Ref-1-data
9,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,123.0,Ref-1-data


In [ ]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = config["amount_type_twins"]

# Apply semantic recoding
df_semantic_recoded = df_cleaned.copy()

    
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")

In [7]:
df_semantic_recod_50 = df_semantic_recod_50.drop(columns=['cement_grade']) # cement grade 38% missing, encodes sames info as cement type
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_semantic_recod_50[fiber_cols] = df_semantic_recod_50[fiber_cols].fillna(0)  # replacing with mean wrong
df_semantic_recod_50.head()

df_semantic_recod_50.to_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv", index=False)

In [8]:
from sklearn.model_selection import GroupShuffleSplit

df = df_semantic_recod_50
target_col = 'cs_28d'
group_col  = 'paper_reference'  

X = df.drop(columns=[target_col, group_col])
y = df[target_col]  
groups = df[group_col]


# 1. carve out test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups=groups))

# 2. carve val from remainder
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=42)
train_idx_rel, val_idx_rel = next(gss2.split(
    X.iloc[train_val_idx], y.iloc[train_val_idx], groups=groups.iloc[train_val_idx]
))

X_train = X.iloc[train_val_idx].iloc[train_idx_rel]
X_val   = X.iloc[train_val_idx].iloc[val_idx_rel]
X_test  = X.iloc[test_idx]
y_train = y.iloc[train_val_idx].iloc[train_idx_rel]
y_val   = y.iloc[train_val_idx].iloc[val_idx_rel]
y_test  = y.iloc[test_idx]

print(f"Train set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns,
                                   handle_unknown='ignore')

print(f"After encoding Training Set Shape: {preprocessor.fit_transform(X_train, y_train).shape}")
print(f"After encoding Validation Set Shape: {preprocessor.transform(X_val).shape}")
print(f"After encoding Test Set Shape: {preprocessor.transform(X_test).shape}")

Train set size: (1461, 33)
Validation set size: (329, 33)
Test set size: (283, 33)
After encoding Training Set Shape: (1461, 60)
After encoding Validation Set Shape: (329, 60)
After encoding Test Set Shape: (283, 60)


In [9]:
# KNN PIPELINE CREATION
knn_pipeline = Pipeline([('preprocessor', preprocessor), ('model', KNeighborsRegressor())])

knn_grid = {
    'model__n_neighbors': list(range(1, 31)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2, 3],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

gs_knn = run_grid_search(knn_pipeline, knn_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'KNN')


GridSearchCV will test 540 combinations for KNN...

BEST HYPERPARAMETERS — KNN
  metric: manhattan
  n_neighbors: 12
  p: 1
  weights: distance
Best CV RMSE: 26.4769

VALIDATION SET PERFORMANCE
RMSE: 38.9877
MAE: 28.5835
MaxAE: 115.6688
R2: 0.3714
Correlation: 0.7556
Mean_Residual: 17.9942
N: 329

TEST SET PERFORMANCE
RMSE: 20.4177
MAE: 15.7183
MaxAE: 65.6889
R2: 0.5376
Correlation: 0.7772
Mean_Residual: -7.7355
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   38.987676  20.417670
            MAE   28.583533  15.718326
          MaxAE  115.668766  65.688938
             R2    0.371375   0.537565
Correlation (R)    0.755585   0.777153
  Mean Residual   17.994223  -7.735481
              N  329.000000 283.000000

TOP 10 COMBINATIONS
   metric  n_neighbors  p  weights  mean_test_score   CV_rmse
manhattan           12  1 distance      -701.027661 26.476927
manhattan           13  1 distance      -702.346273 26.501816
manhattan           12  2 distance     

In [10]:
# SVR PIPELINE CREATION
svr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', SVR(kernel='rbf'))])

svr_grid = {
    'model__C':        [128, 256, 512, 1024],
    'model__epsilon': [0.01, 0.1, 0.5, 1, 3]
}

gs_svr = run_grid_search(svr_pipeline, svr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'SVR')


GridSearchCV will test 20 combinations for SVR...

BEST HYPERPARAMETERS — SVR
  C: 128
  epsilon: 0.01
Best CV RMSE: 30.6932

VALIDATION SET PERFORMANCE
RMSE: 41.8062
MAE: 31.8619
MaxAE: 125.3298
R2: 0.2772
Correlation: 0.7001
Mean_Residual: 19.8351
N: 329

TEST SET PERFORMANCE
RMSE: 23.1699
MAE: 17.7252
MaxAE: 70.1538
R2: 0.4045
Correlation: 0.6841
Mean_Residual: -5.3014
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   41.806206  23.169902
            MAE   31.861928  17.725199
          MaxAE  125.329805  70.153779
             R2    0.277199   0.404493
Correlation (R)    0.700056   0.684136
  Mean Residual   19.835093  -5.301423
              N  329.000000 283.000000

TOP 10 COMBINATIONS
  C  epsilon  mean_test_score   CV_rmse
128     0.01      -942.072985 30.693207
128     0.50      -947.287180 30.778031
128     3.00      -950.849544 30.835848
128     0.10      -951.440154 30.845424
128     1.00      -962.399100 31.022558
256     1.00      -977.236

In [11]:
# NuSVR PIPELINE CREATION
nusvr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', NuSVR(kernel='rbf'))])

nusvr_grid = {
    'model__C':   [128, 256, 512, 1024],
    'model__nu': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

gs_nusvr = run_grid_search(nusvr_pipeline, nusvr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'NuSVR')


GridSearchCV will test 36 combinations for NuSVR...

BEST HYPERPARAMETERS — NuSVR
  C: 128
  nu: 0.3
Best CV RMSE: 30.3323

VALIDATION SET PERFORMANCE
RMSE: 39.4046
MAE: 29.5575
MaxAE: 116.5070
R2: 0.3579
Correlation: 0.7658
Mean_Residual: 18.6591
N: 329

TEST SET PERFORMANCE
RMSE: 22.2897
MAE: 17.4750
MaxAE: 72.1155
R2: 0.4489
Correlation: 0.6941
Mean_Residual: -4.2482
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   39.404581  22.289714
            MAE   29.557479  17.474969
          MaxAE  116.506968  72.115547
             R2    0.357859   0.448879
Correlation (R)    0.765837   0.694111
  Mean Residual   18.659089  -4.248248
              N  329.000000 283.000000

TOP 10 COMBINATIONS
  C  nu  mean_test_score   CV_rmse
128 0.3      -920.051360 30.332348
128 0.2      -928.313485 30.468237
128 0.6      -929.588322 30.489151
256 0.4      -939.846312 30.656913
128 0.1      -942.665393 30.702856
256 0.2      -943.790144 30.721168
128 0.8      -947.57016

In [ ]:
#savinf results to json file

best_params_knn = gs_knn.best_params_
best_params_svr = gs_svr.best_params_
best_params_nusvr = gs_nusvr.best_params_


with open('results.json', 'r') as f:
    results = json.load(f)

results["best_params"] = results.get("best_params", {})
results["best_params"]["best_params_publications_included"] = {
    "knn": best_params_knn,
    "svr": best_params_svr,
    "nusvr": best_params_nusvr,
}

with open('results.json', "w") as f:
    json.dump(results, f, indent=2)

print(results["best_params"])


{'best_params_publications_included': {'knn': {'model__metric': 'manhattan', 'model__n_neighbors': 12, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 128, 'model__epsilon': 0.01}, 'nusvr': {'model__C': 128, 'model__nu': 0.3}}, 'recoded_50': {'knn': {'model__metric': 'manhattan', 'model__n_neighbors': 3, 'model__p': 3, 'model__weights': 'distance'}, 'svr': {'model__C': 512, 'model__epsilon': 3, 'model__gamma': 'scale'}, 'nusvr': {'model__C': 512, 'model__gamma': 'scale', 'model__nu': 0.5}}}
